# VayuNetra Evaluation Harness (Agent 0 & Agent 3)

This notebook evaluates the core multi-agent pipeline:
1. **Latency**: End-to-end signal-to-action time (Target: < 5 min).
2. **Orchestrator Routing Accuracy**: Does it correctly trigger enforcement on spikes?
3. **Enforcement Priority Correlation**: Do prioritized sources correlate with highest population exposure & pollution contribution?
4. **RAG Citation Relevance**: Precision of retrieved regulatory text against source type.

Run this in `DEMO_MODE=true` to test against fixtures, or `DEMO_MODE=false` for live DB traces.

In [ ]:
import os
import time
import json
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns

# Force DEMO_MODE for local testing
os.environ["DEMO_MODE"] = "true"

from agents.graph import run_query
from agents.enforcement import run_enforcement
from rag.retrieve import retrieve_for_enforcement

## 1. Latency & Routing Evaluation
We run the LangGraph orchestrator 10 times and measure end-to-end latency.

In [ ]:
latencies = []
traces = []

print("Running orchestrator 10 times...")
for i in range(10):
    t0 = time.time()
    # Provide focus_cells to guarantee the spike-gate routes to Enforcement
    state = run_query("delhi", focus_cells=["883da1a3a1fffff"])
    latencies.append(state.get("latency_ms", int((time.time() - t0)*1000)))
    traces.append([t["node"] for t in state["trace"]])
    
df_lat = pd.DataFrame({"run": range(1, 11), "latency_ms": latencies})
print(f"\nAverage Latency: {df_lat['latency_ms'].mean():.2f} ms")
print(f"Max Latency: {df_lat['latency_ms'].max()} ms")
print("\nRouting Traces (first 3):")
for t in traces[:3]:
    print(" -> ".join(t))

## 2. Enforcement Priority Distribution
We evaluate how Agent 3 scores recommendations. We expect a healthy spread of scores.

In [ ]:
recs = run_enforcement("delhi")
df_recs = pd.DataFrame([r.to_dict() for r in recs])

print("Enforcement Priorities Generated:", len(df_recs))
display(df_recs[["source_id", "priority_score", "contribution", "pop_exposed", "status"]].head())

if not df_recs.empty:
    plt.figure(figsize=(8, 4))
    sns.scatterplot(data=df_recs, x="contribution", y="priority_score", size="pop_exposed", legend=False, alpha=0.7)
    plt.title("Priority Score vs. PM2.5 Contribution (Bubble size = Pop Exposed)")
    plt.xlabel("PM2.5 Contribution (Share)")
    plt.ylabel("Priority Score")
    plt.grid(True, alpha=0.3)
    plt.show()

## 3. RAG Retrieval Relevance
We check if `retrieve_for_enforcement` fetches the correct CPCB/GRAP rules.

In [ ]:
categories = ["construction_dust", "industrial", "biomass_burning"]
results = []

for cat in categories:
    chunks = retrieve_for_enforcement(cat, top_k=2)
    for c in chunks:
        results.append({
            "category": cat,
            "rule_title": c.title,
            "similarity": c.similarity,
        })
        
df_rag = pd.DataFrame(results)
display(df_rag)